# Beginner tour 1: Meet the data

**Before this notebook:** run `0_setup_check.ipynb` (it checks everything and builds the data).
If you've never written Python, do `../00_how_al_molecular_works/python_from_zero.ipynb` first.

**Who this is for:** you don't need to know Python or machine learning. Each new idea is
explained the first time it appears, in a box called **🐍 Python corner** or **🧪 Science
corner**.

**What this tour is about.** `al-molecular` is a research project that looks for promising
drug molecules among **millions** of candidates while testing as few as possible. Over 8
notebooks you'll run its real code, step by step, on real data.

**This notebook:** look at the real data: molecules, their scores, and the "embeddings" a
computer uses to describe them. Then take a walk around the project's folders.

### How to use a notebook
* Grey boxes are **code cells**. Click one and press **Shift + Enter** to run it. The result appears underneath.
* Run cells **from top to bottom**: later cells use what earlier cells made.
* If things get confusing, use **Restart** (top bar) and run everything again from the top.
* You can't break anything. Change numbers and rerun. That's how you learn.

---
## 1. The problem, in plain words

🧪 **Science corner.** Many medicines work by sticking to a protein in the body and blocking
it. Before buying and testing a molecule in a lab, chemists can ask a computer program to
**dock** it: try to fit the molecule into the protein's pocket and estimate how strongly it
sticks. The answer is a **docking score** in kcal/mol. **More negative = sticks better.**
A score of −12 is excellent; −5 is poor.

Docking takes seconds to minutes per molecule. That's fine for a thousand molecules, but
modern catalogues hold **millions to billions**. So the question this project asks is:

> **Can we find most of the best molecules while docking only a small fraction of them?**

The data here is real. It's 21,044 molecules picked evenly from a 2.1-million-molecule
catalogue (Enamine HTS), each docked against a protein called **thymidylate kinase**.

---
## 2. Loading the data

🐍 **Python corner: `import`.** Python on its own does little. You `import` toolboxes
(called *modules* or *packages*) that other people wrote. `pandas` handles tables and
`numpy` handles big lists of numbers. `tour` is a small helper file next to this notebook
that knows where the data lives.

In [ ]:
import numpy as np
import pandas as pd
from tour import *        # gives us load_mini, draw, AL_REPO, ... (see tour.py)

print("the al-molecular code lives in:", AL_REPO)
print("the mini dataset lives in:     ", MINI_DIR)

🐍 **Python corner: variables and calling a function.** `scores, embeddings = load_mini()`
*calls* the function `load_mini`, which hands back two things. The `=` stores them under
the names `scores` and `embeddings`, so we can use them later. (`=` means "store this
under that name", not "equals" as in maths.)

In [ ]:
scores, embeddings = load_mini()
scores.head()          # .head() shows the first 5 rows of a table

Each row is one molecule. `smiles` is the molecule written as text (explained below), and
`score` is its docking score.

🐍 **Python corner: asking a table questions.** `len(...)` counts things. `table["column"]`
picks a column. Columns can answer questions like `.min()` and `.mean()`.

In [ ]:
print("number of molecules:", len(scores))
print("best score: ", scores["score"].min())
print("worst score:", scores["score"].max())
print("average:    ", round(scores["score"].mean(), 2))

A picture says more. `matplotlib` draws charts. This one is a **histogram**: how many
molecules have each score.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 3))
plt.hist(scores["score"], bins=50)
plt.axvline(scores["score"].nsmallest(100).max(), color="red", linestyle="--", label="the best 100 are left of this line")
plt.xlabel("docking score (kcal/mol), more negative = better")
plt.ylabel("number of molecules")
plt.legend()
plt.show()

Most molecules are average (around −8). Only a thin tail on the left is excellent. **The
goal is to find that tail.** In this tour, "the best molecules" means the **best 100** out
of 21,044, the 0.5% to the left of the red line.

---
## 3. What a molecule looks like to a computer: SMILES

🧪 **Science corner.** **SMILES** writes a molecule as a line of text. Letters are atoms
(`C` carbon, `N` nitrogen, `O` oxygen, `Cl` chlorine, `S` sulfur, `F` fluorine;
lowercase `c` = a carbon in a ring), `=` is a double bond, brackets `( )` are branches,
and numbers mark where a ring closes. `CCO` is ethanol (alcohol). `c1ccccc1` is benzene, a
six-carbon ring.

Let's draw the 8 best molecules. `nsmallest(8, "score")` takes the 8 rows with the smallest
(most negative, i.e. best) scores.

In [ ]:
best8 = scores.nsmallest(8, "score")
draw(best8["smiles"], labels=best8["score"])

And 4 of the worst, for comparison:

In [ ]:
worst4 = scores.nlargest(4, "score")
draw(worst4["smiles"], labels=worst4["score"])

🐍 **Python corner: a `for` loop.** A loop does something once for every item. The indented
lines belong to the loop. `zip(a, b)` walks through two lists side by side. An **f-string**
(`f"..."`) puts values into text with `{ }`.

In [ ]:
for smi, sc in zip(best8["smiles"][:3], best8["score"][:3]):
    print(f"score {sc}:  {smi}")

**Something to notice:** the best molecules are big, with several rings, while the worst
are small. Docking programs are known to favour bigger molecules (more contact with the
protein). Let's check on all 21,044. RDKit, a chemistry toolbox, can count the heavy
(non-hydrogen) atoms in each molecule.

🐍 **Python corner: a list comprehension.** `[f(x) for x in things]` is a one-line loop that
builds a list: "`f(x)` for each `x` in `things`".

In [ ]:
from rdkit import Chem

n_atoms = [Chem.MolFromSmiles(s).GetNumHeavyAtoms() for s in scores["smiles"]]
scores["n_atoms"] = n_atoms          # add it to the table as a new column

plt.figure(figsize=(5, 3.5))
plt.scatter(scores["n_atoms"], scores["score"], s=2, alpha=0.3)
plt.xlabel("number of heavy atoms"); plt.ylabel("docking score")
plt.title(f"correlation = {np.corrcoef(scores['n_atoms'], scores['score'])[0, 1]:.2f}")
plt.show()

A clear downward trend (a correlation of about −0.66): bigger molecules tend to score better
(more negative). But size isn't enough on its own. At any given size, scores still spread
over several kcal/mol. To make better
guesses we need a richer description of each molecule. That's what embeddings are.

---
## 4. Embeddings: describing a molecule with numbers

🧪 **Science corner.** An **embedding** is a long list of numbers that describes a molecule.
It's produced by a big pretrained AI model, called a **backbone** in this project. Each
backbone looks at molecules differently:

| Backbone | Reads the molecule as | Numbers per molecule |
|---|---|---|
| MoLFormer | SMILES text (like a language model reads sentences) | 768 |
| Uni-Mol2 | a 3D shape (atom positions in space) | 768 |
| MHG-GED | a graph (atoms = dots, bonds = lines) | 1,024 |

The project runs each backbone **once** over all molecules (the `embed/` folder does this,
and takes days on GPUs for millions of molecules). It saves the results, so in this tour
they're ready to load.

🐍 **Python corner: dictionaries.** `embeddings` is a **dictionary** (`dict`): pairs of
*key → value*. The keys are backbone names and the values are tables of numbers.
`embeddings["unimol2"]` looks one up.

In [ ]:
print("keys:", list(embeddings.keys()))
for name, table in embeddings.items():
    print(f"{name:10s} shape = {table.shape}   (molecules, numbers per molecule)")

🐍 **Python corner: NumPy arrays.** Each value is a NumPy **array**, a grid of numbers.
`.shape` gives its size: `(21044, 768)` means 21,044 rows (one per molecule) and 768
columns. `array[0]` is row 0 (Python counts from **0**), and `array[0, :5]` is the first 5
numbers of row 0.

In [ ]:
uni = embeddings["unimol2"]
print("molecule 0:", scores["smiles"][0])
print("its first 5 Uni-Mol2 numbers:", uni[0, :5].round(2))

The numbers mean nothing on their own. What matters is that **similar molecules get similar
lists of numbers.** Let's test that. We take the best molecule, find the 4 molecules whose
Uni-Mol2 numbers are *closest* to it, and draw them.

"Closest" here is **cosine similarity**, which measures whether two lists of numbers point
the same way (1 = identical direction, 0 = unrelated). You don't need the formula. Just
read `similarity(...)` as "how alike are these two molecules according to the backbone".

In [ ]:
def similarity(table, i):
    # cosine similarity of row i with every row: dot product of length-1 versions
    unit = table / np.linalg.norm(table, axis=1, keepdims=True)
    return unit @ unit[i]

best_row = scores["score"].idxmin()                # the row number of the best molecule
sim = similarity(uni, best_row)
closest = np.argsort(sim)[::-1][:5]                # argsort sorts; [::-1] flips to biggest first
draw(scores["smiles"][closest], labels=[f"{scores['score'][i]}  (sim {sim[i]:.2f})" for i in closest], per_row=5)

The first picture is the best molecule itself (similarity 1.00). The others are its nearest
neighbours: often similar shapes, and usually good scores too.

**Don't read too much into the similarity numbers themselves.** With Uni-Mol2 they're all
close to 1, even for two *random* molecules (about 0.997), because its 768 numbers mostly
move together. What matters is the **order**: which molecules are *more* similar than others.

Here's the key experiment of this notebook. **Do a molecule's neighbours have scores like
its own?** For the 100 best molecules, compare their neighbours' average score with the
average of all molecules:

In [ ]:
top100 = scores.nsmallest(100, "score").index
for name, table in embeddings.items():
    neighbour_scores = []
    for i in top100[:30]:                            # 30 of the best, to keep it quick
        sim = similarity(table, i)
        nn = np.argsort(sim)[::-1][1:11]             # 10 nearest neighbours, skipping itself
        neighbour_scores.append(scores["score"][nn].mean())
    print(f"{name:10s} neighbours of top molecules average {np.mean(neighbour_scores):6.2f}   "
          f"(all molecules average {scores['score'].mean():.2f})")

The neighbours of the best molecules score clearly better than average, though not equally
for every backbone. Some backbones' notion of "similar" matches docking better than others.
Keep that in mind: it becomes the main story of the project in notebooks 3 and 4.

**This is why the whole project can work.** If similar molecules have similar scores, then
docking a few molecules tells us something about their undocked neighbours, and a model can
learn to *guess* scores from embeddings. That guessing model is called the **surrogate**.

---
## 5. The real project does the same thing

The project's own code loads the scores with a function called `load_oracle()` in
`run_experiment.py`. ("Oracle" means "the thing that knows the true answer": here, the
pre-computed docking scores, looked up instead of re-docking.) Let's use it on our mini
dataset.

🐍 **Python corner: modules are files.** `import run_experiment as rx` loads the file
`run_experiment.py` and names it `rx` for short. Then `rx.load_oracle` is the function
`load_oracle` inside that file.

In [ ]:
import run_experiment as rx

register_mini(rx)            # tell the project about our mini dataset, under the name "MINI"
rx.DATASET = "MINI"          # and choose it
oracle = rx.load_oracle()

`oracle` is a dictionary: **SMILES → score**. Looking up a molecule is like "docking" it, for
free:

In [ ]:
some_molecule = scores["smiles"][0]
print(some_molecule)
print("its docking score:", oracle[some_molecule])
print("dictionary size:", len(oracle))

---
## 6. A walk around the project's folders

🐍 **Python corner: files and folders.** `Path` (from `pathlib`) represents a location on
disk. `.iterdir()` lists what's inside a folder, `.is_dir()` asks "is this a folder?", and
`.name` is the last part of the path.

In [ ]:
what_is_it = {
    "run_experiment.py": "THE main script: runs one active-learning experiment (notebook 2)",
    "surrogates.py":     "the guessing models (surrogates) (notebooks 3, 4)",
    "molpal":            "a library from MIT this project builds on: models, picking rules, ...",
    "data":              "the docking scores (the oracle)",
    "embed":             "scripts that turn molecules into embeddings with each backbone (notebook 7)",
    "adaptive_acq":      "follow-up project: smarter rules for picking molecules (notebook 6)",
    "runs":              "results of every experiment ever run",
    "slurm":             "scripts that send jobs to the university's GPU cluster (notebook 7)",
    "tests":             "automatic checks that the code is right (notebook 8)",
    "docs":              "write-ups of results and findings",
    "reports":           "a polished research report (HTML)",
    "predict_pool_shard_worker.py": "helper that scores one slice of a 100M-molecule pool (notebook 7)",
}
for p in sorted(AL_REPO.iterdir()):
    if p.name in what_is_it:
        kind = "folder" if p.is_dir() else "file  "
        print(f"{kind} {p.name:30s} {what_is_it[p.name]}")

How big are the two most important files? Let's count their lines.

🐍 **Python corner: `with open(...)`.** This opens a file and closes it again automatically
when the indented block ends.

In [ ]:
for name in ["run_experiment.py", "surrogates.py"]:
    with open(AL_REPO / name) as f:
        n_lines = len(f.readlines())
    print(f"{name:20s} {n_lines:,} lines")

About 3,300 lines between them, which is a lot to read cold. This tour reads it in small
pieces, always by running it.

Finally, how big is the **real** data compared with our mini copy?

In [ ]:
import os
big = Path("/N/project/SingleCell_Image/mengjing/enhits_large/embed")
for bb in BACKBONES:
    real_gb = os.path.getsize(big / f"{bb}_embeddings.npy") / 1e9
    mini_mb = os.path.getsize(MINI_DIR / f"{bb}_embeddings.npy") / 1e6
    print(f"{bb:10s} full pool {real_gb:5.1f} GB   |  our mini copy {mini_mb:5.0f} MB")

The full pool doesn't fit comfortably in a laptop's memory, and the largest pool in this
project (D4) is 55× bigger again. Notebook 7 shows how the code copes.

---
## Summary

| Word | Meaning |
|---|---|
| **pool** | all candidate molecules (here 21,044; really 2.1 million to 116 million) |
| **docking score** | how well a molecule sticks to the protein, more negative = better |
| **oracle** | the dictionary of true scores, `{smiles: score}` |
| **SMILES** | a molecule written as text |
| **backbone** | a big pretrained AI model that turns molecules into numbers |
| **embedding** | that list of numbers for one molecule |
| **surrogate** | a small model that learns to *guess* scores from embeddings (next notebooks) |

🐍 **Python you met:** `import`, variables, calling functions, tables (`pandas`), `len`, `for`
loops, `zip`, f-strings, list comprehensions, dictionaries, NumPy arrays and `.shape`,
`Path`, `with open`.

## Try it yourself
Answers, run on the same real data: [solutions/1_solutions.ipynb](solutions/1_solutions.ipynb). Try first!

1. Draw the 8 molecules closest to the **worst** molecule (`scores["score"].idxmax()`). Are they small?
2. In section 4, change `uni` to `embeddings["molformer"]` in the nearest-neighbour cell. Do the neighbours look as similar?
3. What fraction of molecules score better than −10? Hint: `(scores["score"] < -10).mean()`.

**Next:** `2_your_first_experiment.ipynb`, where you run the project's real experiment.